In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
import time
from sklearn.metrics import accuracy_score
from sklearn import svm
from sklearn.neural_network import MLPClassifier


In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
# data = pd.read_excel('dataset_all_original.xlsx') 
data = pd.read_excel('dataset_all_original_ssdp1_2.xlsx') 
# data = pd.read_excel('dataset_all_original_add_nobias.xlsx') 

In [4]:

data = data.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)

In [6]:
data.isnull().sum()

name            0
type            0
vendor          0
DNS           795
mDNS          296
nupnp        4000
open_port     251
ssdp_type     470
ssdp_t        470
ssdp_v        470
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
data.isnull().sum()

name         0
type         0
vendor       0
DNS          0
mDNS         0
nupnp        0
open_port    0
ssdp_type    0
ssdp_t       0
ssdp_v       0
dtype: int64

In [9]:
data.dtypes

name         object
type         object
vendor       object
DNS          object
mDNS         object
nupnp        object
open_port    object
ssdp_type    object
ssdp_t       object
ssdp_v       object
dtype: object

In [10]:
data = data.astype('category')

In [11]:
data.dtypes

name         category
type         category
vendor       category
DNS          category
mDNS         category
nupnp        category
open_port    category
ssdp_type    category
ssdp_t       category
ssdp_v       category
dtype: object

In [12]:
data

,name,type,vendor,DNS,mDNS,nupnp,open_port,ssdp_type,ssdp_t,ssdp_v
0,Chamberlain myQ Garage Opener,Appliances,"The Chamberlain Group, Inc.",taobao.com,_gembucket._udp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NT,urn:schemas-upnp-org:service:WANPPPConnection:1
1,Google Home,Appliances,Google,phoca.cz,_biodex._tcp.local.,NaN,tcp/80,NaN,NaN,NaN
2,iRobot,Appliances,iRobot Corporation,NaN,_andalax._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NT,urn:schemas-upnp-org:device:WANConnectionDevice:1
3,Sonos,Cameras,Sonos Inc,geocities.jp,_alpha._tcp.local.,NaN,NaN,NOTIFY * HTTP/1.1,LOCATION,http://192.168.0.1:1900/igd.xml
4,August Doorbell Cam,Cameras,August Home,NaN,_flowdesk._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,SERVER,Linux/3.14.0 UPnP/1.0 IpBridge/1.33.0
5,iHome,Appliances,SDI Technologies Inc,photobucket.com,_namfix._tcp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,Server,SDK 4.3.0.0 UPnP/1.0 MiniUPnPd/1.6
6,August Doorbell Cam,Cameras,August Home,rediff.com,_veribet._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,01-NLS,1
7,Belkin Netcam,Cameras,Belkin International Inc.,addthis.com,_konklux._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,CONFIGID.UPNP.ORG,1337
8,Bezeq smarthome,Appliances,Bezeq Group,NaN,_y-find._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,CACHE-CONTROL,max-age=100
9,Wyze Cam,Cameras,Wyze Labs,noaa.gov,_trippledex._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,lOCATION,http://192.168.1.1:6352/rootDesc.xml


In [13]:
y = data.drop_duplicates(keep='first')
# y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 
y

,name,type,vendor,DNS,mDNS,nupnp,open_port,ssdp_type,ssdp_t,ssdp_v
0,Chamberlain myQ Garage Opener,Appliances,"The Chamberlain Group, Inc.",taobao.com,_gembucket._udp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NT,urn:schemas-upnp-org:service:WANPPPConnection:1
1,Google Home,Appliances,Google,phoca.cz,_biodex._tcp.local.,NaN,tcp/80,NaN,NaN,NaN
2,iRobot,Appliances,iRobot Corporation,NaN,_andalax._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NT,urn:schemas-upnp-org:device:WANConnectionDevice:1
3,Sonos,Cameras,Sonos Inc,geocities.jp,_alpha._tcp.local.,NaN,NaN,NOTIFY * HTTP/1.1,LOCATION,http://192.168.0.1:1900/igd.xml
4,August Doorbell Cam,Cameras,August Home,NaN,_flowdesk._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,SERVER,Linux/3.14.0 UPnP/1.0 IpBridge/1.33.0
5,iHome,Appliances,SDI Technologies Inc,photobucket.com,_namfix._tcp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,Server,SDK 4.3.0.0 UPnP/1.0 MiniUPnPd/1.6
6,August Doorbell Cam,Cameras,August Home,rediff.com,_veribet._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,01-NLS,1
7,Belkin Netcam,Cameras,Belkin International Inc.,addthis.com,_konklux._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,CONFIGID.UPNP.ORG,1337
8,Bezeq smarthome,Appliances,Bezeq Group,NaN,_y-find._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,CACHE-CONTROL,max-age=100
9,Wyze Cam,Cameras,Wyze Labs,noaa.gov,_trippledex._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,lOCATION,http://192.168.1.1:6352/rootDesc.xml


In [14]:
y = y.astype(str)

In [15]:
y = y.apply(LabelEncoder().fit_transform)

In [16]:
y

,name,type,vendor,DNS,mDNS,nupnp,open_port,ssdp_type,ssdp_t,ssdp_v
0,6,0,28,480,46,0,4,0,10,36
1,10,0,10,403,15,0,5,1,12,8
2,45,0,32,85,8,0,5,0,10,30
3,36,1,26,243,3,0,0,0,6,14
4,2,1,3,85,44,0,0,0,14,7
5,44,0,23,404,78,0,3,0,16,9
6,2,1,3,424,142,0,5,0,0,1
7,3,1,4,93,66,0,0,0,3,2
8,4,0,5,85,153,0,0,0,2,20
9,43,1,30,374,136,0,5,0,19,17


In [17]:
list_data = []
list_data.append(y)
len(list_data)

1

In [18]:
tree_clf = []
time_all = []
time_train = []
time_test = []
accuracy = []

In [19]:
for data in list_data:
    index = np.random.rand(len(data)) < 0.8
    train_data = data[index]
    test_data = data[~index]
    label_data = train_data.type
    train_data = train_data.drop(['type'], axis=1)
    tlabel_data = test_data.type
    test_data = test_data.drop(['type'], axis=1)
    truelabel_data = pd.DataFrame(tlabel_data)
    truelabel_data = truelabel_data.reset_index()
    #time
#     start_time = time.time()
    nn = MLPClassifier(solver='lbfgs', alpha=1e-5,hidden_layer_sizes=(100,), random_state=1)
    linear = svm.SVC(kernel='linear', C=0.1, decision_function_shape='ovo').fit(train_data, label_data)
    rbf = svm.SVC(kernel='rbf', gamma=0.1, C=0.1, decision_function_shape='ovo').fit(train_data, label_data)
    poly = svm.SVC(kernel='poly', degree=3, C=0.1, decision_function_shape='ovo').fit(train_data, label_data)
    sig = svm.SVC(kernel='sigmoid', C=0.1, decision_function_shape='ovo').fit(train_data, label_data)
#     clf = tree.DecisionTreeClassifier()
#     clf = clf.fit(train_data,label_data )
#     tree_clf.append(clf)
    # tree.plot_tree(clf) 
#     time_train.append((time.time() - start_time))
#     start_time_test = time.time()
    linear_pred = linear.predict(test_data)
    poly_pred = poly.predict(test_data)
    rbf_pred = rbf.predict(test_data)
    sig_pred = sig.predict(test_data)
    nn_pred = nn.fit(train_data,label_data )
    
    
    # retrieve the accuracy and print it for all 4 kernel functions
    accuracy_lin = linear.score(test_data, tlabel_data)
    accuracy_poly = poly.score(test_data, tlabel_data)
    accuracy_rbf = rbf.score(test_data, tlabel_data)
    accuracy_sig = sig.score(test_data, tlabel_data)
    accuracy_nn = nn.score(test_data, tlabel_data)
    print("Accuracy Linear Kernel:", accuracy_lin)
    print("Accuracy Polynomial Kernel:", accuracy_poly)
    print("Accuracy Radial Basis Kernel:", accuracy_rbf)
    print("Accuracy Sigmoid Kernel:", accuracy_sig)
    
    print("Accuracy Backpropagation NN:", accuracy_nn)
#     predict_data = clf.predict(test_data)
#     result_label_data = pd.DataFrame(columns = ['type'])
#     result_label_data.type = predict_data
#     time_test.append((time.time() - start_time_test))
#     compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
#     compare.truelabel_data = truelabel_data.type
#     compare.predictedlabel = result_label_data.type
#     accuracy.append(accuracy_score(compare.truelabel_data, compare.predictedlabel))
#     time_all.append(time.time() - start_time)
#     time
#     print("--- %s seconds ---" % (time.time() - start_time))

Accuracy Linear Kernel: 0.5445665445665445
Accuracy Polynomial Kernel: 0.47374847374847373
Accuracy Radial Basis Kernel: 0.4981684981684982
Accuracy Sigmoid Kernel: 0.48717948717948717
Accuracy Backpropagation NN: 0.4981684981684982


In [20]:
print("Accuracy for dataset_gen: ", accuracy[0])
# print("Accuracy for dataset_all_gen: ", accuracy[1])
# print("Accuracy for dataset_all_original: ", accuracy[2])
# print("Accuracy for dataset_jaccard_gen: ", accuracy[3])
# print("Accuracy for dataset_jaccard_original: ", accuracy[4])
# print("Accuracy for dataset_myroom_gen: ", accuracy[5])
# print("Accuracy for dataset_myroom_original: ", accuracy[6])
# print("Accuracy for dataset_university_gen: ", accuracy[7])
# print("Accuracy for dataset_university_original: ", accuracy[8])

IndexError: list index out of range

In [ ]:
print("Time All for dataset_gen: ", time_all[0])
# print("Time All for dataset_all_gen: ", time_all[1])
# print("Time All for dataset_all_original: ", time_all[2])
# print("Time All for dataset_jaccard_gen: ", time_all[3])
# print("Time All for dataset_jaccard_original: ", time_all[4])
# print("Time All for dataset_myroom_gen: ", time_all[5])
# print("Time All for dataset_myroom_original: ", time_all[6])
# print("Time All for dataset_university_gen: ", time_all[7])
# print("Time All for dataset_university_original: ", time_all[8])

In [ ]:
print("Time Train for dataset_gen: ", time_train[0])
# print("Time Train for dataset_all_gen: ", time_train[1])
# print("Time Train for dataset_all_original: ", time_train[2])
# print("Time Train for dataset_jaccard_gen: ", time_train[3])
# print("Time Train for dataset_jaccard_original: ", time_train[4])
# print("Time Train for dataset_myroom_gen: ", time_train[5])
# print("Time Train for dataset_myroom_original: ", time_train[6])
# print("Time Train for dataset_university_gen: ", time_train[7])
# print("Time Train for dataset_university_original: ", time_train[8])

In [ ]:
print("Time Test for dataset_gen: ", time_test[0])
# print("Time Test for dataset_all_gen: ", time_test[1])
# print("Time Test for dataset_all_original: ", time_test[2])
# print("Time Test for dataset_jaccard_gen: ", time_test[3])
# print("Time Test for dataset_jaccard_original: ", time_test[4])
# print("Time Test for dataset_myroom_gen: ", time_test[5])
# print("Time Test for dataset_myroom_original: ", time_test[6])
# print("Time Test for dataset_university_gen: ", time_test[7])
# print("Time Test for dataset_university_original: ", time_test[8])

In [ ]:
print("=================dataset_gen=================")
print("######### Accuracy: "+str(accuracy[0]))

print("######### Time: ")
print("          All: "+str(time_all[0]))
print("          Train: "+str(time_train[0]))
print("          Test: "+str(time_test[0]))
print("######### Tree: ")
tree.plot_tree(tree_clf[0]) 

In [ ]:
print("=================dataset_all_gen_y=================")
print("######### Accuracy: "+str(accuracy[1]))

print("######### Time: ")
print("          All: "+str(time_all[1]))
print("          Train: "+str(time_train[1]))
print("          Test: "+str(time_test[1]))
print("######### Tree: ")
tree.plot_tree(tree_clf[1]) 

In [ ]:
print("=================dataset_all_original_y=================")
print("######### Accuracy: "+str(accuracy[2]))

print("######### Time: ")
print("          All: "+str(time_all[2]))
print("          Train: "+str(time_train[2]))
print("          Test: "+str(time_test[2]))
print("######### Tree: ")
tree.plot_tree(tree_clf[2]) 

In [ ]:
print("=================dataset_jaccard_gen_y=================")
print("######### Accuracy: "+str(accuracy[3]))

print("######### Time: ")
print("          All: "+str(time_all[3]))
print("          Train: "+str(time_train[3]))
print("          Test: "+str(time_test[3]))
print("######### Tree: ")
tree.plot_tree(tree_clf[3]) 

In [ ]:
print("=================dataset_jaccard_original_y=================")
print("######### Accuracy: "+str(accuracy[4]))

print("######### Time: ")
print("          All: "+str(time_all[4]))
print("          Train: "+str(time_train[4]))
print("          Test: "+str(time_test[4]))
print("######### Tree: ")
tree.plot_tree(tree_clf[4]) 

In [ ]:
print("=================dataset_myroom_gen_y=================")
print("######### Accuracy: "+str(accuracy[5]))

print("######### Time: ")
print("          All: "+str(time_all[5]))
print("          Train: "+str(time_train[5]))
print("          Test: "+str(time_test[5]))
print("######### Tree: ")
tree.plot_tree(tree_clf[5]) 

In [ ]:
print("=================dataset_myroom_original_y=================")
print("######### Accuracy: "+str(accuracy[6]))

print("######### Time: ")
print("          All: "+str(time_all[6]))
print("          Train: "+str(time_train[6]))
print("          Test: "+str(time_test[6]))
print("######### Tree: ")
tree.plot_tree(tree_clf[6]) 

In [ ]:
print("=================dataset_university_gen_y=================")
print("######### Accuracy: "+str(accuracy[7]))

print("######### Time: ")
print("          All: "+str(time_all[7]))
print("          Train: "+str(time_train[7]))
print("          Test: "+str(time_test[7]))
print("######### Tree: ")
tree.plot_tree(tree_clf[7])

In [ ]:
print("######### =================dataset_university_original_y=================")
print("######### Accuracy: "+str(accuracy[8]))

print("######### Time: ")
print("          All: "+str(time_all[8]))
print("          Train: "+str(time_train[8]))
print("          Test: "+str(time_test[8]))
print("######### Tree: ")
tree.plot_tree(tree_clf[8])

In [ ]:
label = data.drop(['type'], axis=1)
get_code(clf, label.columns)

In [ ]:
# print("=================dataset_gen=================")
# print("######### Accuracy: "+str(accuracy[0]))

# print("######### Time: ")
# print("          All: "+str(time_all[0]))
# print("          Train: "+str(time_train[0]))
# print("          Test: "+str(time_test[0]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[0]) 

# print("=================dataset_all_gen_y=================")
# print("######### Accuracy: "+str(accuracy[1]))

# print("######### Time: ")
# print("          All: "+str(time_all[1]))
# print("          Train: "+str(time_train[1]))
# print("          Test: "+str(time_test[1]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[1]) 

# print("=================dataset_all_original_y=================")
# print("######### Accuracy: "+str(accuracy[2]))

# print("######### Time: ")
# print("          All: "+str(time_all[2]))
# print("          Train: "+str(time_train[2]))
# print("          Test: "+str(time_test[2]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[2]) 

# print("=================dataset_jaccard_gen_y=================")
# print("######### Accuracy: "+str(accuracy[3]))

# print("######### Time: ")
# print("          All: "+str(time_all[3]))
# print("          Train: "+str(time_train[3]))
# print("          Test: "+str(time_test[3]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[3]) 

# print("=================dataset_jaccard_original_y=================")
# print("######### Accuracy: "+str(accuracy[4]))

# print("######### Time: ")
# print("          All: "+str(time_all[4]))
# print("          Train: "+str(time_train[4]))
# print("          Test: "+str(time_test[4]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[4]) 

# print("=================dataset_myroom_gen_y=================")
# print("######### Accuracy: "+str(accuracy[5]))

# print("######### Time: ")
# print("          All: "+str(time_all[5]))
# print("          Train: "+str(time_train[5]))
# print("          Test: "+str(time_test[5]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[5]) 

# print("=================dataset_myroom_original_y=================")
# print("######### Accuracy: "+str(accuracy[6]))

# print("######### Time: ")
# print("          All: "+str(time_all[6]))
# print("          Train: "+str(time_train[6]))
# print("          Test: "+str(time_test[6]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[6]) 

# print("=================dataset_university_gen_y=================")
# print("######### Accuracy: "+str(accuracy[7]))

# print("######### Time: ")
# print("          All: "+str(time_all[7]))
# print("          Train: "+str(time_train[7]))
# print("          Test: "+str(time_test[7]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[7])

# print("######### =================dataset_university_original_y=================")
# print("######### Accuracy: "+str(accuracy[8]))

# print("######### Time: ")
# print("          All: "+str(time_all[8]))
# print("          Train: "+str(time_train[8]))
# print("          Test: "+str(time_test[8]))
# print("######### Tree: ")
# tree.plot_tree(tree_clf[8])

